# `merge()` / `join()` / `concat()` w pandas — podstawy, narzędzia diagnostyczne i pułapki

**Problem:** złączenia w pandas należą do najczęstszych źródeł *cichych* błędów w analizach — nieoczekiwana duplikacja wierszy po złączeniu 1:many, utrata danych przy złym typie złączenia (`how=`), czy zduplikowany indeks po `concat()`. Żadna z tych sytuacji domyślnie nie rzuca wyjątku — wynik wygląda poprawnie, dopóki ktoś nie zauważy, że liczby się nie zgadzają.

**Porównanie:**
- `merge()` — łączenie po kolumnach (kluczach), odpowiednik SQL `JOIN`.
- `join()` — łączenie po indeksie (skrót dla `merge` z `left_index`/`right_index`).
- `concat()` — sklejanie DataFrame'ów wzdłuż osi (wierszy lub kolumn), bez logiki kluczy.

**Struktura notatnika:** najpierw podstawowa składnia, potem narzędzia diagnostyczne (`indicator`, `validate`), a na końcu pułapki — pokazane tak, żeby było widać, który z wcześniej poznanych parametrów wykryłby dany błąd od razu.

## Setup

In [ ]:
import pandas as pd

employees = pd.DataFrame({
    "employee_id": [1, 2, 3, 4, 5],
    "name": ["Anna", "Bartek", "Celina", "Dawid", "Ewa"],
    "department_id": [10, 20, 10, 30, 20],
})

departments = pd.DataFrame({
    "department_id": [10, 20, 30],
    "department_name": ["Sales", "IT", "HR"],
})

employees

## Sekcja 1 — Podstawy `merge()`

### `on=` vs `left_on=` / `right_on=`

`on=` działa, gdy klucz nazywa się identycznie w obu tabelach. Gdy nazwy się różnią (częste przy danych z różnych źródeł), trzeba wskazać je osobno — a po złączeniu obie kolumny kluczy zostają w wyniku (warto je uporządkować `.drop()`).

In [ ]:
employee_details = pd.DataFrame({
    "id_pracownika": [1, 2, 3, 4, 5],
    "phone": ["111-111-111", "222-222-222", "333-333-333", "444-444-444", "555-555-555"],
})

# Ten sam klucz w obu tabelach -> on=
employees.merge(departments, on="department_id")

# Różne nazwy klucza w obu tabelach -> left_on / right_on
merged_details = employees.merge(employee_details, left_on="employee_id", right_on="id_pracownika")

# Obie kolumny klucza zostają w wyniku - zwykle warto jedną usunąć
merged_details.drop(columns="id_pracownika")

### Typy złączeń — `how=`

`inner` zwraca tylko wiersze z dopasowaniem po obu stronach, `left`/`right` zachowują wszystkie wiersze z jednej strony (uzupełniając brakujące dopasowania `NaN`), `outer` zachowuje wszystko z obu stron.

In [ ]:
# Pracownik bez dopasowanego działu
extra_employee = pd.DataFrame({"employee_id": [6], "name": ["Fabian"], "department_id": [99]})
employees_extra = pd.concat([employees, extra_employee], ignore_index=True)

# Dział bez żadnego przypisanego pracownika
extra_department = pd.DataFrame({"department_id": [40], "department_name": ["Legal"]})
departments_extra = pd.concat([departments, extra_department], ignore_index=True)

for how in ["inner", "left", "right", "outer"]:
    result = employees_extra.merge(departments_extra, on="department_id", how=how)
    print(f"how='{how}': {len(result)} wierszy")

### `suffixes=` — kolizja nazw kolumn spoza klucza

Gdy obie tabele mają kolumnę o tej samej nazwie (inną niż klucz złączenia), pandas domyślnie doda `_x`/`_y` — czytelne dla nikogo poza autorem. Warto nadać suffixy opisujące rzeczywiste znaczenie (np. okresy porównania).

In [ ]:
salary_2025 = pd.DataFrame({"employee_id": [1, 2, 3], "salary": [7000, 8200, 7600]})
salary_2026 = pd.DataFrame({"employee_id": [1, 2, 3], "salary": [7400, 8600, 7900]})

# Domyślne suffixes: _x / _y - nieczytelne w dalszej analizie
comparison_default = salary_2025.merge(salary_2026, on="employee_id")
print(comparison_default.columns.tolist())

# Nazwane suffixy - jasne bez zaglądania do kodu, który je wygenerował
comparison_named = salary_2025.merge(salary_2026, on="employee_id", suffixes=("_2025", "_2026"))
comparison_named

## Sekcja 2 — Narzędzia diagnostyczne

Te dwa parametry nie zmieniają wyniku złączenia — zmieniają to, czy błąd zostanie wykryty od razu, czy dopiero gdy ktoś zauważy nieprawidłowe liczby dalej w analizie.

### `indicator=True` — skąd pochodzi każdy wiersz

Dodaje kolumnę `_merge` z wartością `both`, `left_only` albo `right_only`. Pozwala jawnie zobaczyć, które wiersze nie miały dopasowania — dokładnie to, co przy `how="inner"` (Pułapka 2 w Sekcji 5) znika bez śladu.

In [ ]:
merged_indicator = employees_extra.merge(departments_extra, on="department_id", how="outer", indicator=True)
merged_indicator[["employee_id", "department_id", "_merge"]]

In [ ]:
# Szybki podgląd tylko wierszy bez dopasowania po jednej ze stron
merged_indicator[merged_indicator["_merge"] != "both"]

### `validate=` — jawna deklaracja oczekiwanej kardynalności

Wartości: `"one_to_one"`, `"one_to_many"`, `"many_to_one"`, `"many_to_many"`. Jeśli rzeczywiste dane łamią zadeklarowane założenie (np. spodziewamy się unikalnego klucza po prawej stronie, a jest zduplikowany), `merge()` rzuci wyjątek zamiast po cichu zwrócić więcej wierszy — dokładnie sytuacja z Pułapki 1 w Sekcji 5.

In [ ]:
try:
    employees.merge(departments, on="department_id", validate="many_to_one")
    print("Walidacja przeszła: każdy department_id po prawej stronie jest unikalny")
except Exception as e:
    print(f"Walidacja nie przeszła: {e}")

## Sekcja 3 — `join()` jako skrót po indeksie

`join()` łączy po indeksie zamiast po kolumnie. Wygodne, gdy dane już są zindeksowane po kluczu (np. po wcześniejszym `set_index()`), bo nie trzeba powtarzać `on=`.

In [ ]:
employees_indexed = employees.set_index("department_id")
departments_indexed = departments.set_index("department_id")

joined = employees_indexed.join(departments_indexed, how="left")
joined

## Sekcja 4 — `concat()` — podstawy

### `axis=0` (sklejanie wierszy) vs `axis=1` (sklejanie kolumn)

`axis=0` (domyślne) zakłada te same kolumny w każdej tabeli i dokłada wiersze pod spód. `axis=1` zakłada wspólny/wyrównany indeks i dokłada kolumny obok siebie.

In [ ]:
export_january = pd.DataFrame({"employee_id": [1, 2], "hours_worked": [160, 155]})
export_february = pd.DataFrame({"employee_id": [3, 4], "hours_worked": [162, 158]})

# axis=0: sklejanie wierszy, te same kolumny
stacked = pd.concat([export_january, export_february], axis=0, ignore_index=True)
print(f"axis=0: {stacked.shape}")

# axis=1: sklejanie kolumn obok siebie, wymaga wyrównanego indeksu
salary_col = pd.DataFrame({"salary": [7000, 8200]})
bonus_col = pd.DataFrame({"bonus": [500, 700]})
side_by_side = pd.concat([salary_col, bonus_col], axis=1)
side_by_side

### `keys=` — oznaczenie pochodzenia wiersza

Przy sklejaniu wielu porcji danych (np. eksportów miesięcznych) `keys=` tworzy hierarchiczny indeks, który pozwala później odróżnić, z którego źródła pochodzi dany wiersz — bez dodawania osobnej kolumny ręcznie.

In [ ]:
combined_with_source = pd.concat(
    [export_january, export_february],
    keys=["2026-01", "2026-02"],
    names=["source_month", "row"],
)
combined_with_source

## Sekcja 5 — Pułapki

Te same cztery scenariusze, teraz z odniesieniem do narzędzi z Sekcji 2 i 4 — każdy błąd dało się wykryć od razu, gdyby użyć odpowiedniego parametru.

### Pułapka 1 — zduplikowany klucz po prawej stronie mnoży wiersze

Zabezpieczenie: `validate="many_to_one"` z Sekcji 2b.

In [ ]:
# Symulujemy błąd jakości danych: zduplikowany klucz w tabeli "departments"
departments_dirty = pd.concat([
    departments,
    pd.DataFrame({"department_id": [20], "department_name": ["IT (duplicate record)"]}),
], ignore_index=True)

merged = employees.merge(departments_dirty, on="department_id", how="left")

print(f"employees: {len(employees)} wierszy")
print(f"po merge: {len(merged)} wierszy")  # więcej niż w employees — cicha duplikacja!

# Gdyby użyto validate="many_to_one", poniższe rzuciłoby wyjątek zamiast cicho zwrócić więcej wierszy
try:
    employees.merge(departments_dirty, on="department_id", how="left", validate="many_to_one")
except Exception as e:
    print(f"validate= wykryłby to od razu: {e}")

### Pułapka 2 — zły `how=` cicho usuwa wiersze

Zabezpieczenie: `indicator=True` z Sekcji 2a pokazałby `Fabiana` jako `left_only`, zanim `how="inner"` go usunie.

In [ ]:
inner = employees_extra.merge(departments, on="department_id", how="inner")
left = employees_extra.merge(departments, on="department_id", how="left")

print(f"employees_extra: {len(employees_extra)} wierszy")
print(f"how='inner': {len(inner)} wierszy")  # Fabian zniknął bez ostrzeżenia
print(f"how='left':  {len(left)} wierszy")   # Fabian został, department_name = NaN

### Pułapka 3 — niespójne nazwy kolumn przy `concat()`

`concat()` nie łączy kolumn po znaczeniu, tylko po dokładnej nazwie. Różnica w wielkości liter (typowa przy eksportach z różnych źródeł/miesięcy) tworzy dwie osobne kolumny wypełnione częściowo `NaN` zamiast jednej scalonej.

In [ ]:
export_february_mislabeled = pd.DataFrame({
    "Employee_ID": [3, 4],  # inna wielkość liter niż w export_january!
    "hours_worked": [162, 158],
})

combined_bug = pd.concat([export_january, export_february_mislabeled], ignore_index=True)
combined_bug  # dwie osobne kolumny: employee_id i Employee_ID zamiast jednej

# Zabezpieczenie: normalizacja nazw kolumn przed concat
export_february_fixed = export_february_mislabeled.rename(columns={"Employee_ID": "employee_id"})
combined_fixed = pd.concat([export_january, export_february_fixed], ignore_index=True)
combined_fixed

### Pułapka 4 — domyślny indeks przy `concat()` się duplikuje

Zabezpieczenie: `ignore_index=True` (Sekcja 4a) lub `verify_integrity=True`.

In [ ]:
a = pd.DataFrame({"value": [1, 2]})
b = pd.DataFrame({"value": [3, 4]})

combined_bad = pd.concat([a, b])  # domyślnie zachowuje oryginalne indeksy
print(combined_bad.index.tolist())  # [0, 1, 0, 1] — zduplikowane indeksy
combined_bad.loc[0]  # zwraca DWA wiersze zamiast jednego!

# Zabezpieczenie: verify_integrity=True zatrzymuje concat, jeśli indeks by się zduplikował
try:
    pd.concat([a, b], verify_integrity=True)
except ValueError as e:
    print(f"Wykryto zduplikowany indeks: {e}")

## Podsumowanie

| Pułapka | Objaw | Zabezpieczenie | Sekcja |
|---|---|---|---|
| Zduplikowany klucz po prawej stronie | Więcej wierszy po `merge` niż przed | `validate="many_to_one"` / `"one_to_one"` | 2b |
| Zły `how=` | Ciche zniknięcie wierszy przy `how="inner"` | `indicator=True` przed zawężeniem do `inner` | 2a |
| Kolizja nazw kolumn spoza klucza | Nieczytelne `_x`/`_y` w wyniku | `suffixes=(...)` | 1c |
| Niespójne nazwy kolumn przy `concat` | Dodatkowe kolumny z `NaN` zamiast jednej scalonej | Normalizacja nazw kolumn przed `concat` | 4a |
| Domyślny indeks przy `concat` | Zduplikowane wartości indeksu, błędny `.loc[]` | `ignore_index=True` lub `verify_integrity=True` | 4a |

**Wniosek:** `indicator` i `validate` nie są dodatkami "na wszelki wypadek" — to najtańszy sposób, żeby błąd złączenia ujawnił się w momencie jego popełnienia, a nie kilka kroków dalej w analizie, gdy przyczynę dużo trudniej odtworzyć.